In [1]:
from mlp import Linear, BatchNorm1D, Tanh, Embedding, Layers, FlattenConsecutive
import torch 
import torch.nn.functional as F
from contextensor import ContextTorchTensor, TensorSplit


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/runpy.py", line 197, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/runpy.py", line 87, in _run_code
    exec(code, run_globals)
  File "/Users/vicentearjona/Documents/LLM_practice/.venv/lib/python3.9/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()

# Generate torch tensors from name list

In [2]:
list_to_tensor = ContextTorchTensor(context=8)
list_to_tensor.open(file='names.txt')
X, Y = list_to_tensor.get_tensors()


# Train/validation/test splits

In [3]:
tsplit = TensorSplit(train_size=0.8 , test_size=0.1)
xtrain, xval, xtest, ytrain, yval, ytest = tsplit.split(xs=X, ys=Y)

# Initialize parameters 

In [4]:
dim_emb = 4 # embedding dimension
batch_size = 32 # number examples extracted for each batch. 
dim_hidden = 200 # dimensionality hidden layers. Number of neurons of the hidden layers 
g = 2147483647
vocab_size = list_to_tensor.vocab_size
context = list_to_tensor.context
num_iterations = 200000 
lr = 0.1

## Definition of the tensors

In [5]:
model = Layers([
    Embedding(vocab_size=vocab_size, embeding_dimension=dim_emb, manual_seed=g),
    FlattenConsecutive(number_elements=2),
    Linear(fan_in=2 * dim_emb, fan_out=dim_hidden, generator=g, bias=False), BatchNorm1D(num_features=dim_hidden), Tanh(),
    FlattenConsecutive(number_elements=2),
    Linear(fan_in=2 * dim_hidden, fan_out=dim_hidden, generator=g, bias=False), BatchNorm1D(num_features=dim_hidden), Tanh(),
    FlattenConsecutive(number_elements=2), 
    Linear(fan_in=2 * dim_hidden, fan_out=dim_hidden, generator=g, bias=False), BatchNorm1D(num_features=dim_hidden), Tanh(),
    Linear(fan_in=dim_hidden, fan_out=vocab_size, generator=g, bias=True)
    ])

## Solving initialization issues

In [6]:
with torch.no_grad():
    model.layers[-1].weight *= 0.01
    for layer in model.layers[:-1]:
        if isinstance(layer, Linear):
            if layer.bias is not None:
                layer.bias *= 0.01  
            layer.weight *= 5/3

## Parameters 

In [7]:
parameters = model.parameters()
for p in parameters:
    p.requires_grad = True

In [8]:
print(f'number of parameters = {sum([p.nelement() for p in parameters])}')

number of parameters = 168335


# Forward and backward pass 

In [12]:
# For loop to update the parameters of the NN via gradient descent 
loss_train = [] # object where we store the loss of each iteration 
up_to_data = [] # object where we store the update to date data of each iteration 
for i in range(num_iterations):
# Select minibatch from the complete set 
    ixs = torch.randint(low=0, high=xtrain.shape[0], size=(batch_size,))
    xtr = xtrain[ixs]
    ytr = ytrain[ixs]
# Forward pass. Returns loss
    X = model(xtr)
# Append loss 
    loss_i = F.cross_entropy(input=X, target=ytr)
    loss_train.append(loss_i.log10().item())
# Backward pass. Updates grads
    for layer in model.layers:
        layer.out.retain_grad() 
    for p in parameters: 
        p.grad = None 
    loss_i.backward() # fill grad attributes. Gradient descent for the loss 
# update pass. Recalculates params
    lr = lr if i < int(0.75 * num_iterations) else lr / 100
    for p in parameters:
        p.data += -lr * p.grad
        # Store up to date pass. For each iteration, we store the update to data of each parameter 
        with torch.no_grad():
            up_to_data.append(((lr * p.grad).std() / p.data.std()).log10().item())
    if i % 10000 == 0:
        print(f"Iteration:{i}. Loss:{loss_i.item():.4f}")


Iteration:0. Loss:1.7115
Iteration:10000. Loss:1.5370
Iteration:20000. Loss:2.0548
Iteration:30000. Loss:1.9925
Iteration:40000. Loss:2.0975
Iteration:50000. Loss:1.8828
Iteration:60000. Loss:1.8507
Iteration:70000. Loss:2.4513
Iteration:80000. Loss:2.0655
Iteration:90000. Loss:1.7136
Iteration:100000. Loss:1.6828
Iteration:110000. Loss:2.2063
Iteration:120000. Loss:1.6890
Iteration:130000. Loss:1.5453
Iteration:140000. Loss:1.5739
Iteration:150000. Loss:2.0300
Iteration:160000. Loss:2.2046
Iteration:170000. Loss:1.6494
Iteration:180000. Loss:1.5443
Iteration:190000. Loss:1.8261


In [13]:
with torch.no_grad(): 
    def split_loss(split:str) -> str: 
        X, Y = {
            'train': [xtrain, ytrain], 
            'test': [xtest, ytest], 
            'val': [xval, yval]
        }[split]
        X = model(X)
        # Append loss 
        return f'{split}: loss = {F.cross_entropy(input=X, target=Y)}'

In [14]:
for layer in model.layers:
    layer.train = False 
print(split_loss('train'))
print(split_loss('test'))
print(split_loss('val'))

train: loss = 1.9003078937530518
test: loss = 2.0450055599212646
val: loss = 2.0531139373779297


## Sample from the model 

In [17]:
g = torch.Generator().manual_seed(2147483647 + 10)
out = [] 
for _ in range(10): # Number of words to be sample 
    xstart = [0] * context # initial character. Starting character * context 
    while True: 
        # we pass the current context to the model
        x = model(torch.tensor([xstart]))
        # after iterating over all the layers, the resulting output is the logit tensor
        probs = F.softmax(x,dim=1) # we compute the probabilities 
        ix = torch.multinomial(probs, num_samples=1, generator=g).item() # we extract the next character 
        xstart = xstart[1:] + [ix] 
        out.append(list_to_tensor.itos[ix])
        if ix == 0: 
            break
print(''.join(i for i in out)) # decode and print the generated word 

montalmyah*keerth*hayla*temur*jendrie*caileed*elianna*prenleigh*emmana*leoluwahour*


### Saturation

In [18]:

for i, layer in enumerate(model.layers[:-1]): # note: exclude the output layer
  if isinstance(layer, Tanh):
    t = layer.out
    print('layer %d (%10s): mean %+.2f, std %.2f, saturated: %.2f%%' % (i, layer.__class__.__name__, t.mean(), t.std(), (t.abs() > 0.97).float().mean()*100))

layer 4 (      Tanh): mean -0.00, std 0.84, saturated: 41.62%
layer 8 (      Tanh): mean -0.10, std 0.85, saturated: 44.00%
layer 12 (      Tanh): mean -0.02, std 0.58, saturated: 5.50%


### Gradient statistics 

In [19]:
for i, layer in enumerate(model.layers[:-1]): # note: exclude the output layer
  if isinstance(layer, Tanh):
    t = layer.out.grad
    print('layer %d (%10s): mean %+f, std %e' % (i, layer.__class__.__name__, t.mean(), t.std()))

/var/folders/7b/7nq5283j24db505fwy_chq2r0000gn/T/ipykernel_11921/1108487342.py:3: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more informations. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/build/aten/src/ATen/core/TensorBody.h:494.)
  t = layer.out.grad


AttributeError: 'NoneType' object has no attribute 'mean'

### Gradient to data ratio

In [20]:
# visualize histograms
for i,p in enumerate(parameters):
  t = p.grad
  if p.ndim == 2:
    print('weight %10s | mean %+f | std %e | grad:data ratio %e' % (tuple(p.shape), t.mean(), t.std(), t.std() / p.std()))
   

weight    (27, 4) | mean +0.000000 | std 2.153387e-02 | grad:data ratio 1.424925e-02
weight   (8, 200) | mean -0.000213 | std 8.783240e-03 | grad:data ratio 1.282161e-02
weight (400, 200) | mean -0.000005 | std 3.502781e-03 | grad:data ratio 2.163119e-02
weight (400, 200) | mean -0.000003 | std 3.139849e-03 | grad:data ratio 2.077060e-02
weight  (200, 27) | mean +0.000000 | std 1.647857e-02 | grad:data ratio 7.846357e-02
